### Rebuilt notebook, based on real candidate code found in the author's own files

This notebook replaces an earlier reconstruction (which incorrectly used the repo's
custom `src.models.TransE` PyTorch class). On 2026-11-02 the author located several
real candidate notebooks (`~/Downloads/candidate-ipynbs/`) showing that Scenario 2 runs
actually used **DGL-KE** (`dglke_train`), not the custom class used for Scenario 1.

Two real candidates disagreed on which TransE variant was used:
`goa_reasoner_sc2_TransE.ipynb` uses `TransE_l1`; `GO_KGE_comp.ipynb` uses `TransE_l2`.
Per the author's instruction, this notebook uses **`TransE_l2`**, matching the published
paper's own Methodology equation (1), which is explicitly an L2-norm loss.

No single found file showed a complete baseline-vs-TPR dglke comparison for this specific
scenario/dataset-pair combination -- this notebook combines the data-merge pattern from
`goa_reasoner_sc2_TransE.ipynb` (building a TPR-augmented train.txt by concatenating the
scenario's own valid file) with the `TransE_l2` dglke command from `GO_KGE_comp.ipynb`,
and adds the baseline (unaugmented) run to make the comparison structurally match
Scenario 1's. **This has not been executed.** DGL-KE requires `dgl==0.4.3`, an old,
likely hard-to-install version today, and a GPU -- the author should run this (or confirm
it runs) and check the resulting MRR/Hits@10 against the published Table 2 numbers for
Scenario 2.2 (first- and second-level ancestor depth) on GOA21-22 before treating it as the source of those results.


In [ ]:
import pandas
import src.utils


In [ ]:
code2rel = dict({'F' :'http://purl.obolibrary.org/obo/GO_0003674',
                 'P': 'http://purl.obolibrary.org/obo/GO_0008150',
                 'C' :'http://purl.obolibrary.org/obo/GO_0005575'})

In [ ]:
# Gdrive paths for saving train,test,valid sets -- matches the real candidate
# notebooks and the data confirmed behind Table 1/2 (GO_Chicken source)
ONTO_GDRIVE_PATH = "/content/drive/MyDrive/GOA_Dataset/GO Versions/gene_ontology_"
GOA_DRIVE_PATH   = "/content/drive/MyDrive/GOA_Dataset/GO_Chicken/goa_chicken-"

SAVE_PATH = "/content/drive/MyDrive/GOA_Dataset/"

In [ ]:
spec_version_v1 = "2021-01-01"
spec_version_v2 = "2022-01-01"
dataset_date = "21-22"

### Build the base (unaugmented) train set

In [ ]:
# first version is the train set -- same construction as Scenario 1
triple_df = src.utils.generate_goa_graph(ONTO_GDRIVE_PATH, GOA_DRIVE_PATH, spec_version_v1, 1)
triple_df.rename({0: 'head', 1: 'relation', 2: 'tail'}, axis=1, inplace=True)
triple_df.to_csv('train_baseline.txt', sep='\t', index=False, header=False)
triple_df.to_csv(SAVE_PATH+'train_baseline'+dataset_date+'.txt', sep='\t', index=False, header=False)

### Build the TPR-augmented train set (Scenario 2.2 (first- and second-level ancestor depth))

In [ ]:
# Merge this scenario's own TPR-inferred valid file into the base train set --
# same pattern confirmed in goa_reasoner_sc2_TransE.ipynb for Scenario 2
valid_df = pandas.read_csv(SAVE_PATH+'valid_sc2_l2.txt',
                      header=None,
                      names=["head","relation","tail"],
                      delimiter="\t")
triple_df_tpr = pandas.concat([triple_df, valid_df]).drop_duplicates().reset_index(drop=True)
triple_df_tpr.to_csv('train_tpr.txt', sep='\t', index=False, header=False)
triple_df_tpr.to_csv(SAVE_PATH+'train_tpr'+dataset_date+'.txt', sep='\t', index=False, header=False)

In [ ]:
# Copy this scenario's own valid/test files to the generic names dglke_train expects
import shutil
shutil.copy(SAVE_PATH+'valid_sc2_l2.txt', 'valid.txt')
shutil.copy(SAVE_PATH+'test_sc2_l2.txt', 'test.txt')

### Install dgl-ke (matches the real candidate notebooks' environment)

In [ ]:
!pip3 uninstall dgl -y
!sudo pip3 install dgl==0.4.3
!sudo pip3 install dglke

### Baseline: TransE_l2, no TPR augmentation (Scenario 2.2 (first- and second-level ancestor depth))

In [ ]:
%%time
!cp train_baseline.txt train.txt
!DGLBACKEND=pytorch dglke_train --model_name TransE_l2 --dataset goa --batch_size 1000 --log_interval 100 \
--neg_sample_size 200 --regularization_coef=1e-09 --hidden_dim 400 --gamma 19.9 \
--lr 0.25 --gpu 0 --test -adv --max_step 25000 --save_path /content/baseline \
--data_path /content/predict-KGE-TPR --format raw_udd_hrt --data_files train.txt valid.txt test.txt

### TransE+TPR: TransE_l2, trained on the augmented train set (Scenario 2.2 (first- and second-level ancestor depth))

In [ ]:
%%time
!cp train_tpr.txt train.txt
!DGLBACKEND=pytorch dglke_train --model_name TransE_l2 --dataset goa --batch_size 1000 --log_interval 100 \
--neg_sample_size 200 --regularization_coef=1e-09 --hidden_dim 400 --gamma 19.9 \
--lr 0.25 --gpu 0 --test -adv --max_step 25000 --save_path /content/tpr \
--data_path /content/predict-KGE-TPR --format raw_udd_hrt --data_files train.txt valid.txt test.txt